# Lakeflow Spark Declarative Pipelines

The deck showed the Bronze → Silver → Gold medallion layers. Built by hand, each layer is a separate notebook, and something has to run these notebooks in the right order.

With Lakeflow you declare the same pipeline. Lakeflow Spark Declarative Pipelines (formerly Delta Live Tables) lets you describe *what* each table should contain instead of *how* to compute it. The engine handles the execution order, incremental processing and lineage for you.

> **Exam domains (May-2026 guide):** Data Ingestion and Loading (21%) and Data Transformation and Modeling (22%): streaming tables, materialized views and data-quality expectations (3.6, 3.7). Pipelines are also deployed with bundles in Implementing CI/CD (10%).

> **In class:** the theory below, then a short trainer demo (Section 3), in which the trainer creates, validates and runs the finished `lakeflow_demo` pipeline in the UI. You do the hands-on part in <a href="$../lab/lab_07_lakeflow_pipeline">lab07</a>, where you build the pipeline yourself: bronze and silver with expectations and a quarantine table, SCD Type 1 and Type 2 with `AUTO CDC`, and a small gold star schema. Sections marked **Self-study** are for reading after the course.

## Learning Objectives

After completing this module you will be able to:

- Explain the declarative approach of Lakeflow pipelines compared with imperative Spark code
- Create `STREAMING TABLE` and `MATERIALIZED VIEW` declarations
- Apply data quality `EXPECT` constraints with violation actions (`DROP ROW`, `FAIL UPDATE`, log-only)
- Implement the quarantine pattern for production-grade data quality (in lab07)
- Use `FLOW` declarations to separate table definitions from data sources (self-study)
- Configure `AUTO CDC` for SCD Type 1 and Type 2 processing
- Write Lakeflow pipelines in both SQL and Python (`pyspark.pipelines`)
- Walk through a production-style pipeline (`lakeflow_demo`, self-study)
- Create and run a Lakeflow pipeline in the Databricks UI (in lab07)

## Hands-on Demos for this Module (in `materials/lakeflow/`)

> **Self-study:** skipped in class. Read it after the course: three demos, from simple to advanced.

| # | Demo | Folder | What it shows |
|---|------|--------|---------------|
| 1 | Step by step: build from scratch | `materials/lakeflow/lakeflow_step_by_step/` | A bronze streaming table with `read_files`/Auto Loader (incremental file pickup and checkpoints), AUTO CDC SCD1 vs SCD2 side by side, a gold materialized view with aggregations, then expectations: DROP ROW → FAIL UPDATE → quarantine table. A driver notebook drops file "waves" between updates. |
| 2 | The finished stack | `materials/lakeflow/lakeflow_demo/` | A production-shaped SQL pipeline: full star schema, `CREATE FLOW … INSERT INTO ONCE` backfill and a streaming flow, gold dimensions and fact with expectations. **This is the pipeline deployed by the bundle in LAB 09.** |
| 3 | Metadata-driven | `materials/lakeflow/lakeflow_metadata_driven/` | A config table (1 row = 1 ingestion flow) and one generic source file that generates the whole graph dynamically: add a row, get a table. This is the pattern behind real 100-table ingestion frameworks. *Beyond the exam, for homework or fast finishers.* |

> Each folder has a `README.md` with the exact walkthrough script (steps, talking points, cleanup).


## Section 1: Declarations in SQL and PySpark

The core Lakeflow syntax: table types, constraints, FLOW, AUTO CDC and the PySpark `pyspark.pipelines` API.

> **Pipeline source, shown as text.** The SQL and Python blocks in this section are pipeline source code. The SQL excerpts come from the pipeline files in `materials/lakeflow/`. The Python blocks are illustrative. Since 11 November 2026 (GA) you can also create standalone streaming tables and materialized views from a serverless notebook, for example with `spark.sql("CREATE OR REFRESH STREAMING TABLE ...")`. The course still builds them inside a Lakeflow pipeline, because that is what the exam and `lab_07` use.

### What is Lakeflow?

> **Databricks documentation:** Lakeflow Spark Declarative Pipelines is a framework for creating batch and streaming data pipelines in SQL and Python. You declare the datasets (streaming tables, materialized views) and the transformations; the pipeline manages orchestration, compute, incremental processing, data quality and error handling.
> Source: *[Lakeflow Spark Declarative Pipelines](https://docs.databricks.com/aws/en/ldp/)* (formerly Delta Live Tables, see *[What happened to DLT?](https://docs.databricks.com/aws/en/ldp/concepts/where-is-dlt)*)

Lakeflow Spark Declarative Pipelines (SDP) is a declarative framework for data pipelines.

| Approach | Example | You describe |
|-----------|----------|-------------|
| **Imperative** | `df.write.mode("overwrite")...` | HOW |
| **Declarative** | `CREATE TABLE AS SELECT...` | WHAT |

Key benefits: automatic dependencies, built-in data quality, one model for batch and streaming, automatic recovery, and lineage and monitoring.

### Table Types in Lakeflow

| Type | Usage | Processing |
|-----|--------|-----------|
| **STREAMING TABLE** | Append-only ingestion | Incremental (new rows only) |
| **MATERIALIZED VIEW** | Aggregations, Gold layer | Incremental refresh when possible (serverless), otherwise full recompute |
| **VIEW** | Intermediate logic | Not materialized |

> **Key Point:** A STREAMING TABLE is incremental: it reads new data only and never reprocesses older rows. A MATERIALIZED VIEW always returns the *correct current result* of its query. On serverless pipelines Databricks refreshes it incrementally when the query supports it and when that is cheaper. Otherwise it fully recomputes. On classic compute an MV is always fully recomputed.

> **Databricks documentation (streaming tables):** "A streaming table is a Delta table with additional support for streaming or incremental data processing."
> Source: *[Streaming tables](https://learn.microsoft.com/azure/databricks/ldp/concepts/streaming-tables)*

> **Databricks documentation (materialized view refresh):** "Materialized views attempt an incremental refresh but reprocess all records when necessary to keep the table fully accurate." Incremental refresh is available only on serverless pipelines.
> Source: *[How do pipelines refresh?](https://learn.microsoft.com/azure/databricks/ldp/concepts/refresh)*

<img src="../../../assets/images/nb07-st-vs-mv.png" alt="One pipeline over two updates: the streaming table reads only rows new since the last update, the materialized view recomputes from bronze_orders" width="800">

**STREAMING TABLE vs MATERIALIZED VIEW**

| Feature | STREAMING TABLE | MATERIALIZED VIEW |
|---------|----------------|-------------------|
| Data Source | Streaming (append-only) | Batch or streaming |
| Processing | Incremental (new rows only) | Incremental refresh when possible (serverless), else full recompute |
| Updates | Append-only | Result always matches the query |
| Use Case | Bronze/Silver layers | Gold aggregations |
| Supports AUTO CDC | Yes | No |
| Query Pattern | `STREAM(source)` | Regular `SELECT` |
| Idempotent | Yes (checkpoints) | Yes (declarative result) |

STREAMING TABLEs process only new data (incremental). MATERIALIZED VIEWs keep the full result correct: incrementally when possible, by full recompute when not.

### `STREAM(table)` vs Direct Table Reference

The source syntax inside a Lakeflow pipeline sets the processing mode:

| Syntax | Processing | When to use |
|--------|------------|-------------|
| `FROM STREAM(bronze_orders)` | **Incremental**: only new files/rows | Inside STREAMING TABLE |
| `FROM bronze_orders` | **Snapshot semantics**: the result reflects all rows (the engine may compute it incrementally) | Inside MATERIALIZED VIEW |

Rule of thumb:
- A `STREAMING TABLE` always reads its upstream source with `STREAM(...)`, so it processes new data only.
- A `MATERIALIZED VIEW` reads the source as a static table, so its result always reflects all source rows (incremental refresh on serverless when supported, otherwise full recompute).

> **Exam Tip:** If a pipeline query contains `FROM STREAM(...)`, the target must be declared as a
> `STREAMING TABLE`. A `MATERIALIZED VIEW` that references `STREAM(...)` is invalid.

### STREAMING TABLE with Constraints

> **Summary** (see [Manage data quality with pipeline expectations](https://learn.microsoft.com/azure/databricks/ldp/expectations)): expectations are optional clauses on a streaming table, materialized view or view that check every record. An expectation has a name, a boolean condition and an action for records that fail: warn (default, the row is kept and counted), `ON VIOLATION DROP ROW` (the row is dropped) or `ON VIOLATION FAIL UPDATE` (the update stops).

Constraint actions: `DROP ROW` removes invalid rows, `FAIL UPDATE` fails the pipeline, and no action only logs the violation.

<img src="../../../assets/images/nb07-expectations.png" alt="Five bronze_orders rows through the valid_quantity expectation: warn keeps the bad row, drop removes it, fail stops the update; below, the quarantine pattern" width="800">

**Lakeflow SQL Declarations Reference**

| Statement | Syntax |
|-----------|--------|
| Streaming Table | `CREATE OR REFRESH STREAMING TABLE name (col TYPE, CONSTRAINT c EXPECT (expr) ON VIOLATION action)` |
| Materialized View | `CREATE OR REFRESH MATERIALIZED VIEW name AS SELECT ...` |
| Expectation: warn only | *(omit ON VIOLATION)*: all rows kept, violation logged in metrics |
| Expectation: drop rows | `ON VIOLATION DROP ROW` |
| Expectation: fail update | `ON VIOLATION FAIL UPDATE` |
| Read upstream incremental | `FROM STREAM(table_name)` |
| Read upstream full scan | `FROM table_name` |
| Pipeline parameter | `'${parameter_name}'` |

The three actions as they appear in `materials/lakeflow/lakeflow_demo/transformations/` (excerpts from `02_silver/silver_orders.sql` and `03_gold/fact_sales.sql`):

```sql
-- silver_orders: drop the row
CONSTRAINT valid_order_id EXPECT (order_id IS NOT NULL)
  ON VIOLATION DROP ROW,

-- fact_sales: stop the update
CONSTRAINT valid_order_id EXPECT (order_id IS NOT NULL)
  ON VIOLATION FAIL UPDATE,

-- fact_sales: warn only (the row stays, the violation is counted in the metrics)
CONSTRAINT valid_amounts EXPECT (gross_amount >= 0 AND net_amount >= 0)
```

### Expectations: Quarantine Pattern

Instead of dropping or failing, you can route invalid rows to a **quarantine table** and keep them for investigation. Rows that pass all rules go to silver, and the rest go to the quarantine table. Valid and quarantined rows together add up to the bronze row count. You build this pattern yourself in <a href="$../lab/lab_07_lakeflow_pipeline">lab07</a>. The reference code is in `materials/lakeflow/lakeflow_step_by_step/transformations/05_quarantine.py`.

### Gold: dimensions and a fact table

From `materials/lakeflow/lakeflow_demo/transformations/03_gold/dim_customer.sql` (excerpt, `...` marks omitted lines):

```sql
CREATE OR REFRESH MATERIALIZED VIEW dim_customer
AS
SELECT
  row_number() OVER (ORDER BY customer_id) AS customer_key,
  customer_id,
  first_name,
  last_name,
  -- ...
FROM silver_customers
WHERE __END_AT IS NULL

UNION ALL
-- ... one 'UNKNOWN' row with customer_key -1
```

From `materials/lakeflow/lakeflow_demo/transformations/03_gold/fact_sales.sql` (excerpt, `...` marks omitted lines):

```sql
CREATE OR REFRESH STREAMING TABLE fact_sales
(
  CONSTRAINT valid_order_id EXPECT (order_id IS NOT NULL) 
    ON VIOLATION FAIL UPDATE,
  -- ...
)
AS
SELECT
  -- surrogate keys to dimensions
  o.order_id,
  -- ...
  COALESCE(c.customer_key, -1)            AS customer_key,
  COALESCE(p.product_key, -1)             AS product_key,
  -- ...
  COALESCE(d.date_key, o.order_date_key)  AS order_date_key,
  -- ...
  o.gross_amount,
  o.discount_amount,
  o.net_amount,
  -- ...
FROM STREAM(silver_orders) o
LEFT JOIN dim_customer c
  ON o.customer_id = c.customer_id
LEFT JOIN dim_product p
  ON o.product_id = p.product_id
-- ...
```

### What is FLOW?

> **Self-study:** skipped in class.

> **Summary** (see [Flows](https://learn.microsoft.com/azure/databricks/ldp/concepts/flows)): a flow is the query that writes data into a target such as a streaming table. Several flows can append to the same streaming table, for example to combine sources or to add a one-time backfill next to a live stream.

A FLOW separates the table definition from the data source. Key capabilities:
1. Multiple sources write to one table (for example a backfill and a stream).
2. CDC with automatic SCD through `AUTO CDC`.
3. `INSERT INTO ONCE` for a one-time backfill.
4. `INSERT INTO` for continuous streaming.

From `materials/lakeflow/lakeflow_demo/transformations/01_bronze/bronze_orders.sql` (excerpt, `...` marks omitted lines):

```sql
-- 1. RAW ORDERS (BRONZE) – target streaming table
CREATE OR REFRESH STREAMING TABLE bronze_orders;

-- 1b. Stream from orders_stream_*.json (continuous data flow)
CREATE FLOW bronze_orders_stream
AS INSERT INTO bronze_orders BY NAME
SELECT
  -- ...
FROM STREAM read_files(
  '${order_path}/stream/orders_stream_*.json',
  format           => 'json',
  inferColumnTypes => true
);
```

### Backfill + Streaming Pattern

> **Self-study:** skipped in class.

**FLOW Syntax Reference**

| Statement | Syntax |
|-----------|--------|
| Empty target table | `CREATE OR REFRESH STREAMING TABLE bronze_orders;` |
| One-time backfill | `CREATE FLOW name AS INSERT INTO ONCE target BY NAME SELECT ... FROM read_files(...)` |
| Continuous stream | `CREATE FLOW name AS INSERT INTO target BY NAME SELECT ... FROM STREAM read_files(...)` |
| CDC flow (SCD1) | `CREATE FLOW name AS AUTO CDC INTO target FROM STREAM src KEYS (key) SEQUENCE BY ts STORED AS SCD TYPE 1` |
| CDC flow (SCD2) | `CREATE FLOW name AS AUTO CDC INTO target FROM STREAM src KEYS (key) SEQUENCE BY ts STORED AS SCD TYPE 2` |

From `materials/lakeflow/lakeflow_demo/transformations/01_bronze/bronze_orders.sql` (excerpt, `...` marks omitted lines):

```sql
-- 1. RAW ORDERS (BRONZE) – target streaming table
CREATE OR REFRESH STREAMING TABLE bronze_orders;

-- 1a. Backfill from historical orders file (batch, one-time)
CREATE FLOW bronze_orders_backfill
AS 
INSERT INTO ONCE bronze_orders BY NAME
SELECT
  order_id,
  customer_id,
  -- ...
  'batch'                            AS source_system,
  -- ...
FROM read_files(
  '${order_path}/orders_batch.json',
  format           => 'json',
  inferColumnTypes => true
);

-- 1b. Stream from orders_stream_*.json (continuous data flow)
CREATE FLOW bronze_orders_stream
AS INSERT INTO bronze_orders BY NAME
SELECT
  order_id,
  customer_id,
  -- ...
  'stream'                           AS source_system,
  -- ...
FROM STREAM read_files(
  '${order_path}/stream/orders_stream_*.json',
  format           => 'json',
  inferColumnTypes => true
);
```

### What is SCD (Slowly Changing Dimension)?

A Slowly Changing Dimension (SCD) is a strategy for handling changes in dimension data over time.

| Type | Strategy | Result | Use Case |
|------|---------|--------|----------|
| **SCD Type 0** | Retain original | Always "Warsaw" | Reference data that never changes |
| **SCD Type 1** | Overwrite | Only "Kraków" | Error corrections, non-historical data |
| **SCD Type 2** | Track full history | Both records with timestamps | Audit, compliance, historical analysis |
| **SCD Type 3** | Add column | `current_city=Kraków`, `previous_city=Warsaw` | Single step back only |

> In Lakeflow, `STORED AS SCD TYPE 1` overwrites and `STORED AS SCD TYPE 2` tracks the full history with `__START_AT` / `__END_AT`.

### AUTO CDC for SCD Type 2

> **Summary** (see [The AUTO CDC APIs](https://learn.microsoft.com/azure/databricks/ldp/cdc)): `AUTO CDC` replaces `APPLY CHANGES` (same syntax) and applies change events from a source to a target streaming table, stored as SCD Type 1 (update in place, no history) or SCD Type 2 (history kept in `__START_AT` / `__END_AT`).

AUTO CDC compares new records by `KEYS` and detects changes. With SCD2 it closes the old record (`__END_AT`) and inserts the new one. With SCD1 it overwrites the record.

> **AUTO CDC vs Change Data Feed:** Change Data Feed (CDF, see module <a href="$../../day1/demo/03_delta_lake">03 Delta Lake</a>) records row-level changes inside a Delta table. `AUTO CDC` brings change events from a source into a target table.

<img src="../../../assets/images/nb07-auto-cdc-scd.png" alt="Two change events for one customer and the resulting rows with AUTO CDC as SCD Type 1 and as SCD Type 2" width="800">

**AUTO CDC Syntax Reference**

| Element | Syntax / Value |
|---------|---------------|
| Target table | `CREATE OR REFRESH STREAMING TABLE silver_customers (customer_id STRING, ..., __START_AT TIMESTAMP, __END_AT TIMESTAMP)` |
| CDC Flow | `CREATE FLOW name AS AUTO CDC INTO target FROM STREAM source` |
| Business key | `KEYS (customer_id)` |
| Change order | `SEQUENCE BY ingestion_ts` |
| SCD Type 1 (overwrite) | `STORED AS SCD TYPE 1` |
| SCD Type 2 (history) | `STORED AS SCD TYPE 2` |
| Query current records | `WHERE __END_AT IS NULL` |

From `materials/lakeflow/lakeflow_demo/transformations/02_silver/silver_customers.sql` (excerpt, `...` marks omitted lines):

```sql
-- 3. Create the target SCD2 table
CREATE OR REFRESH STREAMING TABLE silver_customers (
  customer_id        STRING,
  first_name         STRING,
  last_name          STRING,
  email              STRING,
  -- ...
  city               STRING,
  -- ...
  __START_AT         TIMESTAMP,
  __END_AT           TIMESTAMP
);

-- 4. Create the AUTO CDC flow for SCD2
CREATE FLOW silver_customers_scd2_flow
AS AUTO CDC INTO silver_customers
FROM STREAM bronze_customers
KEYS (customer_id)
SEQUENCE BY ingestion_ts
STORED AS SCD TYPE 2;
```

### PySpark Declarations

Lakeflow pipelines can also be defined in Python with the `pyspark.pipelines` decorators. A streaming table is a function decorated with `@dp.table` that returns a streaming DataFrame. A materialized view uses `@dp.materialized_view` with a batch read (`spark.read.table`).

```python
from pyspark import pipelines as dp
from pyspark.sql.functions import *

# STREAMING TABLE
@dp.table(
    name="bronze_customers",
    comment="Raw customers from CSV"
)
def bronze_customers():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .load(spark.conf.get("customer_path"))
            .select(
                "*",
                col("_metadata.file_path").alias("source_file_path"),
                current_timestamp().alias("load_ts")
            )
    )
```

> **Self-study:** skipped in class. The materialized view counterpart:

```python
# MATERIALIZED VIEW: a batch read needs @dp.materialized_view (@dp.table is for streaming tables)
@dp.materialized_view(name="dim_customer")
def dim_customer():
    return (
        spark.read.table("silver_customers")
            .filter(col("__END_AT").isNull())
            .select("customer_id", "first_name", "last_name")
    )
```

### PySpark: Expectations

> **Self-study:** skipped in class.

Decorators: `@dp.expect` only logs the violation, `@dp.expect_or_drop` drops the record, and `@dp.expect_or_fail` fails the pipeline.

```python
@dp.table(name="silver_orders")
@dp.expect_or_drop("valid_order_id", "order_id IS NOT NULL")
@dp.expect_or_drop("valid_customer", "customer_id IS NOT NULL")
@dp.expect_or_drop("valid_quantity", "quantity IS NOT NULL AND quantity <> 0")
@dp.expect("valid_unit_price", "unit_price IS NOT NULL AND unit_price >= 0")
def silver_orders():
    return (
        spark.readStream.table("bronze_orders")
            .select(
                "order_id",
                "customer_id",
                col("order_datetime").cast("timestamp").alias("order_ts"),
                (col("quantity") * col("unit_price")).alias("gross_amount")
            )
    )
```

### PySpark: Reusable Expectations & Quarantine

> **Self-study:** skipped in class.

Use `expect_all` / `expect_all_or_drop` to apply one dictionary of rules to several datasets. Combine them with the quarantine pattern for production-grade data quality.

| Decorator | Behavior |
|---|---|
| `@dp.expect_all(rules)` | Log all violations (warn) |
| `@dp.expect_all_or_drop(rules)` | Drop rows failing ANY rule |
| `@dp.expect_all_or_fail(rules)` | Fail pipeline on ANY violation |

```python
from pyspark import pipelines as dp
from pyspark.sql.functions import expr

# Reusable rules dictionary, shared across datasets
quality_rules = {
    "valid_order_id": "(order_id IS NOT NULL)",
    "valid_quantity":   "(quantity IS NOT NULL AND quantity <> 0)",
    "valid_unit_price": "(unit_price IS NOT NULL AND unit_price >= 0)",
    "valid_date":     "(order_datetime IS NOT NULL)",
}

# Build quarantine expression: NOT(rule1 AND rule2 AND ...)
quarantine_expr = "NOT({0})".format(" AND ".join(quality_rules.values()))

# Intermediate table: tags each row and logs rule metrics
# private (formerly temporary): not published to the catalog
@dp.table(private=True, partition_cols=["is_quarantined"])
@dp.expect_all(quality_rules)  # logs pass/fail counts for all rules
def orders_quarantine():
    return (
        spark.readStream.table("bronze_orders")
        .withColumn("is_quarantined", expr(quarantine_expr))
    )

# Valid records go to downstream processing
@dp.temporary_view
def valid_orders():
    return spark.read.table("orders_quarantine").filter("is_quarantined=false")

# Invalid records go to quarantine for investigation
@dp.temporary_view
def quarantine_orders():
    return spark.read.table("orders_quarantine").filter("is_quarantined=true")
```

### PySpark: AUTO CDC

> **Self-study:** skipped in class.

The Python equivalent of SQL AUTO CDC uses `dp.create_auto_cdc_flow()` to define the SCD logic in code.

```python
from pyspark import pipelines as dp

# Define the target table
dp.create_streaming_table(
    name="silver_customers",
    schema="""
        customer_id STRING,
        first_name STRING,
        city STRING,
        __START_AT TIMESTAMP,
        __END_AT TIMESTAMP
    """
)

# Define the CDC flow
dp.create_auto_cdc_flow(
    target="silver_customers",
    source="bronze_customers",
    keys=["customer_id"],
    sequence_by="ingestion_ts",
    stored_as_scd_type=2  # or 1
)
```

### What's New in Lakeflow (2025-2026)

> **Self-study:** skipped in class.

| Feature | Status | Description |
|---------|--------|-------------|
| **`pyspark.pipelines`** | GA | New Python module replacing `import dlt` |
| **`expect_all` dictionaries** | GA | Reusable rules across multiple datasets |
| **Quarantine pattern** | Recommended | Route invalid rows to a separate table for investigation |
| **Sinks API** | GA | Write pipeline output to Kafka, Event Hubs, external Delta tables |
| **ForEachBatch** | Preview (Dec 2025) | Process streams as micro-batches, write to arbitrary targets |
| **Auto Loader File Events** | GA (Dec 2025) | Efficient file discovery without cloud notification setup |
| **Lakeflow System Tables** | Preview (Dec 2025) | Enhanced monitoring: pipeline metrics, job duration breakdown |

> **Key trend:** Lakeflow pipelines are evolving toward serverless-first compute, declarative sinks for external targets, and deeper observability via system tables.

## Section 2: Pipeline Materials in lakeflow_demo

> **Self-study:** skipped in class.

The DEA repo contains a production-style RetailHub pipeline in `materials/lakeflow/lakeflow_demo/transformations/`.
It is written in SQL, the most readable format for the declarative approach. All three medallion layers use the RetailHub scenario (customers, orders, products, stores, payment methods).

| File | Layer | Object / pattern |
|------|-------|------------------|
| `01_bronze/bronze_customers.sql` | Bronze | `STREAMING TABLE` + `STREAM read_files()` (CSV, only `*.csv`) |
| `01_bronze/bronze_orders.sql` | Bronze | `STREAMING TABLE` + 2 `FLOW`s: `INSERT INTO ONCE` backfill + continuous stream |
| `01_bronze/bronze_products.sql` | Bronze | `MATERIALIZED VIEW` + `read_files()` (Parquet snapshot) |
| `02_silver/silver_customers.sql` | Silver | `STREAMING TABLE` + `AUTO CDC` → SCD Type 2 |
| `02_silver/silver_orders.sql` | Silver | `STREAMING TABLE` + 5 expectations (`DROP ROW`) + computed measures |
| `02_silver/silver_products.sql` | Silver | `MATERIALIZED VIEW`: casting, dedup per `product_id`, unknown member |
| `03_gold/dim_customer.sql` | Gold | `MATERIALIZED VIEW`: current SCD2 snapshot + unknown member `-1` |
| `03_gold/dim_date.sql` | Gold | `MATERIALIZED VIEW`: date dimension |
| `03_gold/dim_product.sql` | Gold | `MATERIALIZED VIEW`: product dimension |
| `03_gold/dim_store.sql` | Gold | `MATERIALIZED VIEW`: store dimension (with unknown member) |
| `03_gold/dim_payment_method.sql` | Gold | `MATERIALIZED VIEW`: payment method dimension |
| `03_gold/fact_sales.sql` | Gold | `STREAMING TABLE`: fact table with dimension lookups + expectations |

> **Note:** Section 3 shows how to configure and run this pipeline end-to-end in the Databricks UI.

### Bronze Layer: Two Ingestion Patterns

`bronze_customers.sql` uses a simple Auto Loader read (one continuous source):

```sql
CREATE OR REFRESH STREAMING TABLE bronze_customers
AS SELECT
  customer_id, first_name, last_name, email, city, customer_segment,
  _metadata.file_path              AS source_file_path,
  _metadata.file_modification_time AS ingestion_ts,
  current_timestamp()              AS load_ts
FROM STREAM read_files('${customer_path}', format => 'csv',
                        header => true, inferColumnTypes => true,
                        fileNamePattern => '*.csv');   -- the folder also holds an .xlsx file
```

`bronze_orders.sql` uses the FLOW pattern (historical backfill plus a live stream):

```sql
CREATE OR REFRESH STREAMING TABLE bronze_orders;

CREATE FLOW bronze_orders_backfill
AS INSERT INTO ONCE bronze_orders BY NAME
SELECT ..., 'batch' AS source_system
FROM read_files('${order_path}/orders_batch.json', format => 'json');

CREATE FLOW bronze_orders_stream
AS INSERT INTO bronze_orders BY NAME
SELECT ..., 'stream' AS source_system
FROM STREAM read_files('${order_path}/stream/orders_stream_*.json', format => 'json');
```

Key patterns:
- `_metadata.file_path` and `_metadata.file_modification_time` add automatic data provenance at ingest.
- `INSERT INTO ONCE` runs exactly once to load the historical data and never reprocesses it.
- `INSERT INTO` runs on every pipeline trigger and picks up only new files (based on checkpoints).
- The empty `STREAMING TABLE` declaration separates the schema contract from the data sources.

> **Why Gold shows more sales than the batch file:** the `orders_stream_*.json` files repeat orders that are already in `orders_batch.json`, so Bronze, Silver and `fact_sales` hold these orders twice and the totals in Gold are higher. The pipeline works as written, and this demo does not remove duplicates.

### Silver Layer: Quality Gates and SCD Type 2

`silver_customers.sql` uses AUTO CDC to keep the full change history (SCD2):

```sql
CREATE OR REFRESH STREAMING TABLE silver_customers (
  customer_id STRING, first_name STRING, city STRING, ...,
  __START_AT  TIMESTAMP,
  __END_AT    TIMESTAMP
);

CREATE FLOW silver_customers_scd2_flow
AS AUTO CDC INTO silver_customers
FROM STREAM bronze_customers
KEYS (customer_id)
SEQUENCE BY ingestion_ts
STORED AS SCD TYPE 2;
```

`silver_orders.sql` has 5 quality expectations and the business transformations (excerpt, `...` marks omitted lines):

```sql
CREATE OR REFRESH STREAMING TABLE silver_orders
(
  CONSTRAINT valid_order_id EXPECT (order_id IS NOT NULL)
    ON VIOLATION DROP ROW,
  CONSTRAINT valid_customer EXPECT (customer_id IS NOT NULL)
    ON VIOLATION DROP ROW,
  CONSTRAINT valid_product EXPECT (product_id IS NOT NULL)
    ON VIOLATION DROP ROW,
  CONSTRAINT valid_quantity EXPECT (quantity IS NOT NULL AND quantity <> 0)
    ON VIOLATION DROP ROW,
  CONSTRAINT valid_unit_price EXPECT (unit_price IS NOT NULL AND unit_price >= 0)
    ON VIOLATION DROP ROW
)
AS
SELECT
  -- ...
  (o.quantity * o.unit_price)                                   AS gross_amount,
  (o.quantity * o.unit_price * o.discount_percent / 100.0)      AS discount_amount,
  (o.quantity * o.unit_price)
    - (o.quantity * o.unit_price * o.discount_percent / 100.0)  AS net_amount,
  -- ...
  CASE 
    WHEN o.quantity < 0 OR o.total_amount < 0 
      THEN 1 ELSE 0 
  END AS is_return,
  -- ...
FROM STREAM(bronze_orders) o;
```

Key patterns:
- The 5 constraints form a cascading quality gate: any row that fails one of them is dropped before Gold.
- `__START_AT` is set on insert. `__END_AT` is set when a newer version arrives (`NULL` = current record).
- Business metrics (`gross_amount`, `net_amount`) are computed in Silver, so all downstream Gold tables share one source of truth.

### Gold Layer: Fact Table with Dimension Lookups

`fact_sales.sql` is a STREAMING TABLE that joins Silver with all dimensions:

```sql
CREATE OR REFRESH STREAMING TABLE fact_sales (
  CONSTRAINT valid_order_id     EXPECT (order_id IS NOT NULL)
    ON VIOLATION FAIL UPDATE,
  CONSTRAINT valid_customer_key EXPECT (customer_key IS NOT NULL AND customer_key >= -1),
  CONSTRAINT valid_product_key  EXPECT (product_key IS NOT NULL AND product_key >= -1),
  CONSTRAINT valid_amounts      EXPECT (gross_amount >= 0 AND net_amount >= 0)
)
AS SELECT
  o.order_id,
  COALESCE(c.customer_key, -1)           AS customer_key,   -- -1 = unknown customer
  COALESCE(p.product_key,  -1)           AS product_key,
  COALESCE(d.date_key, o.order_date_key) AS order_date_key,
  o.quantity, o.gross_amount, o.net_amount,
  o.net_amount - (COALESCE(p.unit_cost, 0) * o.quantity) AS profit_amount
FROM STREAM(silver_orders) o
LEFT JOIN dim_customer c  ON o.customer_id = c.customer_id
LEFT JOIN dim_product   p ON o.product_id  = p.product_id
LEFT JOIN dim_store     s ON o.store_id    = s.store_id
LEFT JOIN dim_date      d ON o.order_date_key = d.date_key;
```

Key patterns:
- `FAIL UPDATE` on `valid_order_id`: a null order ID in Gold is a critical failure, so the pipeline stops immediately.
- `COALESCE(key, -1)`: a fact row whose dimension key is not found gets the unknown-member key `-1` (`dim_customer` and `dim_product` have an explicit `UNKNOWN` row with key `-1`). A streaming fact row is written only once. If the dimension row arrives later, facts that are already written are **not** re-resolved.
- `fact_sales` is a `STREAMING TABLE` (not a `MATERIALIZED VIEW`), so it processes each Silver batch incrementally.
- The profit calculation uses `p.unit_cost` from `dim_product`, so the fact is enriched across layers without re-reading Bronze.

> **SCD1 vs SCD2 in Gold:** `dim_customer` queries `silver_customers WHERE __END_AT IS NULL` (current snapshot).  
> For historical analysis, query `silver_customers` directly with `__START_AT` / `__END_AT` filters.

## Section 3: Building a Lakeflow Pipeline in the UI

> **Trainer demo (in class):** the trainer runs these steps live before the lab, with the finished `lakeflow_demo` pipeline: create, dry run, run and the pipeline graph. Then, in <a href="$../lab/lab_07_lakeflow_pipeline">lab07</a>, you create your own pipeline in the Lakeflow Pipelines Editor.

The steps below create, configure, validate and run the `lakeflow_demo` pipeline on **serverless** compute: pipeline creation, source assets, configuration, dry run and execution. Verified on the training workspace.

### Step 1: Create the Pipeline

In the left sidebar, go to **Jobs & Pipelines** → **Create** → **ETL pipeline**.

<img src="../../../assets/images/ss-07-1a.png" width="800">

The editor creates the pipeline at once. It is named `New Pipeline <date time>` and has a sample file `transformations/my_transformation.py` in your user folder. Open **Settings** and rename the pipeline with the pencil next to **Pipeline name** (or click the title; both work), for example `lakeflow_pipeline_<your_name>`. The two ways open different dialogs. The pencil dialog has **Also rename root folder** ticked by default. If you type the name in the title instead, a dialog asks **Also rename the root folder?** with **Cancel** and **Rename**. Any choice is fine, because step 3 points the pipeline at the course files.

<img src="../../../assets/images/ss-07-1b.png" width="500">

### Step 2: Default Catalog and Schema

In **Settings** → **Default location for data assets** → **Edit catalog and schema**, select your catalog `retailhub_<your_name>` and the schema `<your_name>_lakeflow` for the pipeline output tables. If the schema does not exist yet, type `<your_name>_lakeflow` in the selector and choose **Create schema**. The **Create a new schema** dialog opens with an empty name field: type `<your_name>_lakeflow` again, click **Create**, then **Save**. A dedicated schema per participant avoids name conflicts in a shared workspace. The screenshots show the trainer's catalog.

<img src="../../../assets/images/ss-07-2.png" width="500">

### Step 3: Point the Pipeline at the Source Code

In **Settings** → **Code assets** → **Configure paths**, pick the folders with **Browse** in your course folder (the dialog needs absolute workspace paths):
- Set **Pipeline root folder** to `…/materials/lakeflow/lakeflow_demo`.
- Set **Source code paths** to its `transformations` subfolder (all `.sql` files in `01_bronze/`, `02_silver/`, `03_gold/`).

<img src="../../../assets/images/ss-07-3.png" width="500">

The sample file from step 1 is outside these paths, so the editor marks it **Excluded** and the pipeline no longer runs it. You can delete it.

### Step 4: Compute and Configuration

Open **Settings**:

<img src="../../../assets/images/ss-07-4.png" width="500">

- Under **Compute**, keep **Serverless** (the default for new pipelines). You do not size a cluster, it starts fast and you pay per use. You need classic pipeline compute (fixed or autoscaling workers) only when serverless is not available. Its VMs take several minutes to start.
- Under **Configuration** → **Add configuration** (**Edit configuration** once keys exist), set these parameters. The SQL files read them as `${key}`:

| Key | Value |
|-----|-------|
| `customer_path` | `/Volumes/retailhub_<your_name>/default/datasets/customers` |
| `order_path` | `/Volumes/retailhub_<your_name>/default/datasets/orders` |
| `product_path` | `/Volumes/retailhub_<your_name>/default/datasets/products/products.parquet/` |

Click **Save**. The editor builds the pipeline graph from the source files.

### Step 5: Validate the Pipeline with a Dry Run

> **Dry run** is in [Public Preview](https://learn.microsoft.com/azure/databricks/ldp/updates#check-a-pipeline-for-errors-without-waiting-for-tables-to-update).

Open the **Run pipeline** dropdown and choose **Dry run** to validate the pipeline without writing data. The dry run parses every declaration, resolves the dependencies between datasets and checks that the sources can be analyzed. On serverless it takes under a minute. It catches typical failures such as a missing configuration key, a typo in a table name or an empty source folder (`CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE`). The streaming source `${order_path}/stream/` must already hold the `orders_stream_*.json` files, which the course setup copies. Otherwise the flow `bronze_orders_stream` fails with that error.

<img src="../../../assets/images/ss-07-5.png" width="800">

After a successful validation, the pipeline DAG shows all tables, dependencies and data flow paths from bronze to gold. If the editor still shows a code tab, close it or open the **Pipeline graph** tab. Light-bulb hints on the materialized views (for example a non-Delta source or `current_timestamp()`) are suggestions about incremental refresh, not errors.

### Step 6: Execute Pipeline

Click **Run pipeline**. The first run processes all source data. Later runs process only **new** files for streaming tables (checkpoints) and refresh the materialized views. **Full table refresh** resets the streaming checkpoints and rebuilds every table from scratch. Use it only when you need to reprocess everything, for example after a breaking schema change.

<img src="../../../assets/images/ss-07-6.png" width="800">

A successful run shows every dataset with a green status: `bronze_*` → `silver_*` → `dim_*` materialized views and the `fact_sales` streaming table.

> **Result:** The Lakeflow pipeline ran end to end, from raw source ingestion (Bronze) through validated transformations (Silver) to business-ready dimensions and facts (Gold). Lakeflow managed the dependencies and enforced the data quality rules.

## APPLY CHANGES INTO: legacy name of AUTO CDC

`APPLY CHANGES INTO` is the Delta Live Tables name of `AUTO CDC`, with the same clauses (`KEYS`, `SEQUENCE BY`, `STORED AS SCD TYPE`). Recognize it on the exam, and write `AUTO CDC` in new code.

## Summary

Key concepts covered in this module.

| Topic | Key Concept | Key Terms |
|---|---|---|
| **SCD Type 1** | Overwrite, no history | `AUTO CDC ... STORED AS SCD TYPE 1` |
| **SCD Type 2** | Track history | `__START_AT`, `__END_AT`, `AUTO CDC` |
| **STREAMING TABLE** | Append-only, incremental | `CREATE STREAMING TABLE`, `STREAM()` |
| **MATERIALIZED VIEW** | Always-correct result; incremental refresh on serverless when possible | `CREATE MATERIALIZED VIEW` |
| **FLOW** (self-study) | Separate source from table | `INSERT INTO ONCE` (backfill) |
| **Expectations** | Data quality constraints | `DROP ROW`, `FAIL UPDATE`, warn-only |
| **Quarantine** (lab07) | Route invalid records separately | quarantine table, same rules as silver |
| **Sinks** (self-study) | Write to external targets | ForEachBatch, Kafka, Event Hubs |

### Resources

- [Lakeflow Spark Declarative Pipelines docs](https://docs.databricks.com/aws/en/ldp/)
- [Medallion Architecture](https://www.databricks.com/glossary/medallion-architecture)
- [AUTO CDC (SCD Type 1/2)](https://docs.databricks.com/aws/en/ldp/cdc)

← <a href="$./06_delta_optimization">06 — Delta Optimization</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$../../day3/demo/08_job_orchestration">08 — Job Orchestration →</a>